### Zaras dritter Fehler – Lösung Schritt für Schritt

*30. Bundeswettbewerb Informatik (2011), 1. Runde, Aufgabe 4*

Zara Zackig muss sich viele geheime Sätze für telefonische Identitätsfeststellungen merken. Sie schreibt sie nur verschlüsselt auf: Sie möchte zum Beispiel den Satz „Ohne Liebe keine Wahrheit“ verschlüsseln. Leer- und Satzzeichen ignoriert sie. Nun sucht sie sich eine beliebige Stelle in ihrem Lieblingsbuch aus. Sie findet die Buchstaben ihres Satzes (als Untersequenz) im gewählten Text, zählt die Abstände und erhält so eine Zahlenfolge, die sie sich notiert. Für den Satz oben ergibt sich mit der Textstelle „Ganz einfach. So geweckt und temperamentvoll …“ die Folge

    13, 34, 7, 13, 11, 3, 9, 58, 1, 93, 4, 1, 1, 1, 4, 21, 7, 3, 23, 5, 5, 15

Welche Zeile im Buch als Startpunkt dient, will sie sich aber auswendig merken!

#### Aufgabe 4

Eines Tages will Zara einen ihrer Sätze entschlüsseln und erkennt mit Grauen, dass sie die richtige Zeile im Buch vergessen hat. Die notierte Zahlenfolge lautet:

    1, 30, 21, 8, 5, 19, 46, 3, 22, 20, 74, 9, 1, 86, 1, 12, 12, 8, 1, 13, 295, 4, 25, 96, 2, 2, 327, 37,
    1, 1, 9, 34, 11, 3, 3, 9, 1, 5, 2, 13, 20, 79, 1, 1, 1, 1, 4, 55, 1, 17, 1, 1, 1, 4, 2, 104, 12, 235, 37

Ermittle für diese und alle weiteren Zahlenfolgen, die zusammen mit dem Text von Zaras Lieblingsbuch auf www.bundeswettbewerb-informatik.de abgelegt sind, welcher Satz bzw. deutsche Klartext dahinter steckt.

**Zu den Daten:** `beispieldaten/EffiBriest_01.txt` ist Zaras Lieblingsbuch („Effi Briest“ von Theodor Fontane, Text von Project Gutenberg). Die Dateien `zahlenfolge0.txt` bis `zahlenfolge9.txt`, `zahlenfolgeA.txt` und `zahlenfolgeB.txt` enthalten je eine Zahlenfolge, durch Leerzeichen getrennt. `zahlenfolge0.txt` ist das Beispiel „Ohne Liebe keine Wahrheit“.

## Was dir bei dieser Aufgabe hilft

**Algorithmische Kenntnisse**
- **Alle Möglichkeiten durchprobieren (Brute Force):** Der Satz fängt in irgendeiner Zeile an. Es gibt nur rund 10 000 Zeilen, also erzeugen wir zu jeder Zeile den Kandidaten und prüfen alle. Das ist schnell genug.
- **Präfixsummen:** Die Folge enthält Abstände. Wenn man sie fortlaufend aufaddiert (`13, 34, 7` wird zu `13, 47, 54`), erhält man die Positionen der Buchstaben im Text.
- **Kandidaten bewerten:** Der richtige Satz besteht aus deutschen Wörtern, ein falscher Kandidat nur aus Buchstabensalat. Eine Zahl („Wie viel des Kandidaten lässt sich in Wörter zerlegen?“) macht die Kandidaten vergleichbar.
- **Zerlegung eines Textes in Wörter durch dynamische Programmierung:** Man berechnet von hinten nach vorn für jede Stelle die beste Bewertung des Rests und merkt sich sie. So probiert man nicht jede Zerlegung einzeln durch (Backtracking würde oft dasselbe Reststück mehrfach prüfen).
- **Datenstruktur Menge:** Ob ein Buchstabenblock ein Wort ist, fragt man tausende Male ab. Eine Menge (`set`) beantwortet das sehr schnell, eine Liste müsste jedes Mal durchsucht werden.

**Nützliche Funktionen und Bibliotheken**
- `str.lower()`, `str.isalpha()`: Kleinbuchstaben und „ist ein Buchstabe“ (auch für ä, ö, ü, ß).
- `str.split()`: zerlegt einen Text an Leerzeichen in Wörter.
- `set(...)`, `x in menge`: Wörterbuch als Menge.
- `enumerate(datei, 1)`: Zeilen mit laufender Nummer lesen.
- `max(...)` und `sorted(..., reverse=True)` für Tupel `(Punkte, ...)`: Python vergleicht Tupel von links nach rechts, das größte erste Element gewinnt.
- `str.find(zeichen, ab)`: sucht ein Zeichen ab einer Stelle (für die Gegenprobe, wenn wir selbst verschlüsseln).

## Die Idee in Worten

1. **Text vorbereiten:** Zaras Verfahren ignoriert alles außer Buchstaben. Wir machen aus dem Buch deshalb eine lange Folge von Kleinbuchstaben und merken uns, wo jede Zeile beginnt. Leere Zeilen gibt es dabei nicht mehr (sie würden nur doppelte Kandidaten liefern).
2. **Zu jedem Zeilenanfang einen Kandidaten bauen:** Zahlenfolge zu Präfixsummen, dann die Buchstaben an diesen Positionen sammeln.
3. **Wörterbuch:** Wir haben kein deutsches Wörterbuch. Die Hinweise erlauben aber, die Wörter aus „Effi Briest“ selbst zu verwenden. Wir sammeln sie in einer Menge.
4. **Kandidaten bewerten:** Wie viel des Kandidaten lässt sich aus Wörtern des Buches zusammensetzen? Längere Wörter zählen deutlich mehr als kurze.
5. **Der Kandidat mit der besten Bewertung gewinnt.** Wir prüfen, ob der Abstand zum zweitbesten groß genug ist, und lösen damit alle Zahlenfolgen.
6. **Gegenprobe:** Wir verschlüsseln selbst einen Satz und entschlüsseln ihn wieder.

## Schritt 1: Das Buch einlesen und säubern

Aus jeder Zeile behalten wir nur die Buchstaben, in Kleinbuchstaben. `zeile.lower()` macht Kleinbuchstaben, `z.isalpha()` ist für Buchstaben (auch Umlaute und `ß`) wahr, für Leerzeichen, Satzzeichen, Ziffern und Anführungszeichen falsch. Zeilen, die danach leer sind, lassen wir weg. Für jede übrig gebliebene Zeile merken wir uns

- die **Nummer der Zeile in der Datei** (zum Anzeigen),
- an welcher **Stelle im langen Text** sie beginnt (zum Rechnen).

Es gibt zwei Texte: `zeilen` (je Zeile gesäubert) und `text` (alle Zeilen hintereinander).

In [ ]:
def saeubere(zeile):
    # Nur Buchstaben behalten, alles klein
    return ''.join(z for z in zeile.lower() if z.isalpha())


def lies_buch(dateiname):
    zeilen = []          # gesäuberte, nichtleere Zeilen
    dateizeilen = []     # zugehörige Zeilennummer in der Datei (ab 1)
    with open(dateiname, encoding='utf-8') as f:
        for nr, zeile in enumerate(f, 1):
            sauber = saeubere(zeile)
            if sauber != '':
                zeilen.append(sauber)
                dateizeilen.append(nr)
    return zeilen, dateizeilen


zeilen, dateizeilen = lies_buch('beispieldaten/EffiBriest_01.txt')
text = ''.join(zeilen)

# Wo beginnt jede Zeile im langen Text?
anfaenge = []
pos = 0
for z in zeilen:
    anfaenge.append(pos)
    pos += len(z)

print('Nichtleere Zeilen:', len(zeilen))
print('Buchstaben im Text:', len(text))
print('Verschiedene Zeichen:', ''.join(sorted(set(text))))
print('Erste Zeile (gesäubert):', zeilen[0])
print('Die ersten 60 Buchstaben:', text[:60])

Nichtleere Zeilen: 9875
Buchstaben im Text: 484440
Verschiedene Zeichen: abcdefghijklmnopqrstuvwxyzßàâäçèéôöü
Erste Zeile (gesäubert): theprojectgutenbergebookofeffibriestbytheodorfontane
Die ersten 60 Buchstaben: theprojectgutenbergebookofeffibriestbytheodorfontanethiseboo


Es bleiben 9875 Zeilen mit insgesamt 484 440 Buchstaben, die 26 Buchstaben, `ä ö ü ß` und ein paar französische Buchstaben wie `é` oder `ç` (Effi und ihre Gäste sprechen manchmal Französisch). (Der Gutenberg-Vorspann und die Lizenz am Ende gehören zur Datei dazu, auch dort kann Zara beginnen.)

## Schritt 2: Zahlenfolge und Zeile geben einen Kandidaten

Beginnt der Satz in einer Zeile, die an der Stelle `start` im langen Text anfängt, dann zählt die erste Zahl `13` ab dem Zeilenanfang: Der erste Buchstabe ist der 13. Buchstabe ab dem Zeilenanfang, also `text[start + 13 - 1]`. Die nächste Zahl `34` zählt ab diesem Buchstaben weiter, der zweite Buchstabe steht 34 Stellen hinter dem ersten. Mit der **laufenden Summe** (Präfixsumme) `13`, `47`, `54`, … ist der `k`-te Buchstabe also `text[start + summe_k - 1]`.

Der Abstand `1` heißt „direkt der nächste Buchstabe“. Ein Abstand `0` kann nicht vorkommen, weil der nächste Buchstabe hinter dem vorigen liegen muss (Untersequenz).

In [ ]:
def lies_folge(dateiname):
    with open(dateiname, encoding='utf-8') as f:
        return [int(x) for x in f.read().split()]


def kandidat(folge, start):
    # Buchstaben, die entstehen, wenn der Satz am Zeilenanfang 'start' (Stelle im langen Text) beginnt
    ergebnis = ''
    summe = 0
    for abstand in folge:
        summe += abstand
        stelle = start + summe - 1
        if stelle >= len(text):
            return None            # läuft über das Buchende hinaus
        ergebnis += text[stelle]
    return ergebnis


beispiel = lies_folge('beispieldaten/zahlenfolge0.txt')
print('Folge:', beispiel)

# Alle Zeilen durchgehen: in welcher beginnt "ohne"?
for nr in range(len(zeilen)):
    k = kandidat(beispiel, anfaenge[nr])
    if k is not None and k.startswith('ohne'):
        print('Zeile', dateizeilen[nr], 'der Datei:', k)

Folge: [13, 34, 7, 13, 11, 3, 9, 58, 1, 93, 4, 1, 1, 1, 4, 21, 7, 3, 23, 5, 5, 15]
Zeile 1321 der Datei: ohneliebekeinewahrheit


Das Beispiel funktioniert: In genau einer Zeile entsteht „ohneliebekeinewahrheit“. Wir dürfen den Satz hier schon lesen, weil wir ihn kennen. Bei den unbekannten Folgen gibt es aber 9875 Kandidaten und wir müssen den richtigen **automatisch** finden. Dafür brauchen wir ein Wörterbuch.

## Schritt 3: Das Wörterbuch aus dem Buch selbst

Ein Wörterbuch mit allen deutschen Wörtern haben wir nicht. Die Lösungshinweise schlagen vor, notfalls die Wörter des Buches selbst zu nehmen. Das geht so: Wir ersetzen im Originaltext alles, was kein Buchstabe ist, durch ein Leerzeichen und zerlegen den Text mit `split()` in Wörter. Eine **Menge** (`set`) behält jedes Wort nur einmal und prüft `wort in woerter` sehr schnell.

Kurze Wörter (ein oder zwei Buchstaben, z. B. „a“, „is“, „en“) kommen in jedem Buchstabensalat zufällig vor und würden schlechte Kandidaten aufwerten. Deshalb behalten wir nur Wörter **ab 3 Buchstaben**. Auch sehr lange Wörter lassen wir weg, weil es solche Stücke im Salat praktisch nie gibt (und die Suche langsamer wird).

In [ ]:
with open('beispieldaten/EffiBriest_01.txt', encoding='utf-8') as f:
    roh = f.read()

nur_buchstaben = ''.join(z if z.isalpha() else ' ' for z in roh.lower())
woerter = set(w for w in nur_buchstaben.split() if 3 <= len(w) <= 20)
print('Verschiedene Wörter (3 bis 20 Buchstaben):', len(woerter))
print('Sind "liebe", "wahrheit", "keine" dabei?', 'liebe' in woerter, 'wahrheit' in woerter, 'keine' in woerter)
print('"ohne"? ', 'ohne' in woerter, ' "pferden"?', 'pferden' in woerter)

Verschiedene Wörter (3 bis 20 Buchstaben): 11693
Sind "liebe", "wahrheit", "keine" dabei? True True True
"ohne"?  True  "pferden"? True


## Schritt 4: Wie „deutsch“ ist ein Kandidat?

Wir zerlegen einen Kandidaten in Wörter aus dem Wörterbuch. Es sollen möglichst viele Buchstaben in langen Wörtern stecken. Buchstaben, die zu keinem Wort gehören, zählen nichts. Als **Punktzahl** nehmen wir die Summe der **Quadrate der Wortlängen**: Ein Wort mit 8 Buchstaben bringt 64 Punkte, zwei Wörter mit je 4 Buchstaben nur 32. So gewinnt die Zerlegung in lange Wörter, und zufällige kurze Treffer fallen kaum ins Gewicht.

Wie findet man die beste Zerlegung? Wir gehen von hinten nach vorn: `best[i]` ist die beste Punktzahl für den Rest des Kandidaten ab Stelle `i`. Für Stelle `i` gibt es zwei Möglichkeiten:
- der Buchstabe an Stelle `i` gehört zu keinem Wort: `best[i] = best[i + 1]`,
- ab Stelle `i` beginnt ein Wort der Länge `l`: `best[i] = l² + best[i + l]`.

Wir nehmen das Maximum. `best[0]` ist dann die Punktzahl des ganzen Kandidaten. Weil jeder Rest nur einmal berechnet wird, ist das viel schneller als Backtracking.

In [ ]:
def bewerte(kand):
    # beste Punktzahl der Zerlegung in Wörter (von hinten nach vorn)
    n = len(kand)
    best = [0] * (n + 1)
    for i in range(n - 1, -1, -1):
        b = best[i + 1]                       # Buchstabe i gehört zu keinem Wort
        for laenge in range(3, min(20, n - i) + 1):
            if kand[i:i + laenge] in woerter:
                b = max(b, laenge * laenge + best[i + laenge])
        best[i] = b
    return best[0]


def zerlege(kand):
    # dasselbe, liefert aber die Zerlegung als Text mit Leerzeichen (unbekannte Buchstaben stehen einzeln)
    n = len(kand)
    best = [0] * (n + 1)
    wahl = [1] * (n + 1)                      # Länge des Wortes ab Stelle i, 1 = kein Wort
    for i in range(n - 1, -1, -1):
        best[i] = best[i + 1]
        for laenge in range(3, min(20, n - i) + 1):
            if kand[i:i + laenge] in woerter and laenge * laenge + best[i + laenge] > best[i]:
                best[i] = laenge * laenge + best[i + laenge]
                wahl[i] = laenge
    teile = []
    i = 0
    while i < n:
        teile.append(kand[i:i + wahl[i]])
        i += wahl[i]
    return ' '.join(teile)


print(bewerte('ohneliebekeinewahrheit'), zerlege('ohneliebekeinewahrheit'))
print(bewerte('xqzvwpyfkdjhgsbnmtlr'), zerlege('xqzvwpyfkdjhgsbnmtlr'))
print(bewerte('ganzeinfachsogeweckt'), zerlege('ganzeinfachsogeweckt'))

130 ohne liebe keine wahrheit
0 x q z v w p y f k d j h g s b n m t l r
114 ganz einfach s o geweckt


Der Beispielsatz wird in vier Wörter zerlegt und erreicht die volle Punktzahl (alle 22 Buchstaben gehören zu Wörtern: `4² + 5² + 5² + 8² = 130`). Ein zufälliger Buchstabenhaufen bekommt dagegen kaum Punkte.

## Schritt 5: Die beste Zeile finden

Wir erzeugen zu jedem Zeilenanfang den Kandidaten, bewerten ihn und merken uns den besten. Zum Vergleich merken wir auch die **zweitbeste** Punktzahl: Wenn der Sieger weit vor allen anderen liegt, ist das Ergebnis sehr sicher.

In [ ]:
def entschluessele(folge):
    # Gibt (Punkte, Zeile in der Datei, Kandidat, zweitbeste Punktzahl) zurück
    bewertet = []
    for nr in range(len(zeilen)):
        k = kandidat(folge, anfaenge[nr])
        if k is not None:
            bewertet.append((bewerte(k), nr, k))
    bewertet.sort(reverse=True)
    punkte, nr, k = bewertet[0]
    return punkte, dateizeilen[nr], k, bewertet[1][0]


punkte0, zeile0, k0, zweit0 = entschluessele(beispiel)
print('Beispiel: Punkte', punkte0, '| Zeile', zeile0, '|', zerlege(k0), '| zweitbester Kandidat:', zweit0, 'Punkte')

Beispiel: Punkte 130 | Zeile 1321 | ohne liebe keine wahrheit | zweitbester Kandidat: 100 Punkte


Für das Beispiel gewinnt der richtige Kandidat mit allen 130 Punkten, der zweitbeste hat 100 Punkte. Der ist also nicht weit weg: Auch ein Buchstabensalat enthält manchmal zufällig ein längeres Wort. Wie selten das ist, sehen wir uns gleich an. (Die Zeile ist die Nummer in der Textdatei, mit allen Leerzeilen mitgezählt.)

## Schritt 6: Alle Zahlenfolgen

Zuerst die Folge aus der Aufgabenstellung. Sie steht als `zahlenfolge1.txt` in den Beispieldaten, wir prüfen das mit den Zahlen aus dem Aufgabenblatt. Dann lösen wir alle 13 Folgen (`zahlenfolge0` bis `9`, `A`, `B`: zwölf Dateien, dazu die Folge der Aufgabe, die mit `zahlenfolge1` übereinstimmt) und geben aus: die Zeile, die Bewertung des Siegers im Vergleich zum zweitbesten Kandidaten und den Klartext, so wie unsere Zerlegung ihn liefert.

In [ ]:
aufgabenfolge = [1, 30, 21, 8, 5, 19, 46, 3, 22, 20, 74, 9, 1, 86, 1, 12, 12, 8, 1, 13, 295, 4, 25, 96, 2, 2, 327, 37,
                 1, 1, 9, 34, 11, 3, 3, 9, 1, 5, 2, 13, 20, 79, 1, 1, 1, 1, 4, 55, 1, 17, 1, 1, 1, 4, 2, 104, 12, 235, 37]
print('Folge der Aufgabe = zahlenfolge1.txt?', aufgabenfolge == lies_folge('beispieldaten/zahlenfolge1.txt'))
print()

ergebnisse = {}
for name in '0123456789AB':
    folge = lies_folge('beispieldaten/zahlenfolge' + name + '.txt')
    punkte, zeile, k, zweit = entschluessele(folge)
    ergebnisse[name] = (zeile, k, punkte, zweit)
    print('zahlenfolge' + name, '(%d Zahlen)' % len(folge), '| Zeile', zeile, '| Punkte', punkte, 'gegen', zweit)
    print('   ', zerlege(k))

Folge der Aufgabe = zahlenfolge1.txt? True

zahlenfolge0 (22 Zahlen) | Zeile 1321 | Punkte 130 gegen 100
    ohne liebe keine wahrheit
zahlenfolge1 (59 Zahlen) | Zeile 391 | Punkte 378 gegen 127
    das denken soll man den pferden überlassend i e h abenden größeren kopf
zahlenfolge2 (63 Zahlen) | Zeile 467 | Punkte 341 gegen 127
    wenn d u irgend etwas verloren hast nimm and u hättest e seinem armen gegeben
zahlenfolge3 (65 Zahlen) | Zeile 47 | Punkte 264 gegen 143
    der e l l e n bogen ist dem mund e nahe dennoch kann man nicht selbst hinein bei ß e n
zahlenfolge4 (39 Zahlen) | Zeile 347 | Punkte 157 gegen 88
    wovon das herz voll ist davon geht der mund über
zahlenfolge5 (42 Zahlen) | Zeile 67 | Punkte 127 gegen 93
    j e v e r d o r b e n erde r staat desto mehr gesetz e hat e r
zahlenfolge6 (63 Zahlen) | Zeile 580 | Punkte 346 gegen 141
    erinnere dich mensch das s d u aus s tau b b i stund z u s tau b zurückkehren wirst
zahlenfolge7 (54 Zahlen) | Zeile 768 | Punkte 204 ge

Bei allen Folgen liegt der Sieger vor dem zweitbesten Kandidaten, am knappsten bei `zahlenfolge5` (127 gegen 93 Punkte). Unsere Zerlegung ist nicht perfekt: Das Wörterbuch kennt nur Wörter, die in „Effi Briest“ vorkommen, und die Quadrate der Wortlängen bevorzugen lange Wörter. Deshalb steht dort z. B. „überlassend“ und anschließend einzelne Buchstaben, obwohl der Satz „überlassen, die haben“ heißt. Den Klartext erkennt man trotzdem sofort. Die Aufgabe verlangt auch nicht, den Satz vom Programm fehlerfrei zerlegen zu lassen, sondern nur, ihn zu finden.

## Schritt 7: Die Klartexte

Wir lesen die Sätze ab und schreiben sie mit Groß- und Kleinschreibung und Satzzeichen auf. Zur Kontrolle säubern wir jeden Klartext mit `saeubere` und vergleichen ihn mit dem gefundenen Kandidaten. Stimmt alles, steht davor `True`.

In [ ]:
klartexte = {
    '0': 'Ohne Liebe keine Wahrheit',
    '1': 'Das Denken soll man den Pferden überlassen, die haben den größeren Kopf.',
    '2': 'Wenn du irgendetwas verloren hast, nimm an, du hättest es einem Armen gegeben.',
    '3': 'Der Ellenbogen ist dem Munde nahe, dennoch kann man nicht selbst hineinbeißen.',
    '4': 'Wovon das Herz voll ist, davon geht der Mund über.',
    '5': 'Je verdorbener der Staat, desto mehr Gesetze hat er.',
    '6': 'Erinnere dich, Mensch, dass du aus Staub bist und zu Staub zurückkehren wirst.',
    '7': 'Wer den Aal hält bei dem Schwanz, dem bleibt er weder halb noch ganz.',
    '8': 'Schlechte Handwerker klagen über ihr Werkzeug.',
    '9': 'Wenn man durch Zweifel läuft, ist der Weg zum Himmel lang.',
    'A': 'Jede Dummheit leidet am Ekel vor sich selbst.',
    'B': 'Ein guter Spruch ist die Wahrheit eines ganzen Buches in einem einzigen Satz.',
}
for name in '0123456789AB':
    zeile, k, punkte, zweit = ergebnisse[name]
    print(saeubere(klartexte[name]) == k, '| Folge', name, '| Zeile', zeile, '|', klartexte[name])

True | Folge 0 | Zeile 1321 | Ohne Liebe keine Wahrheit
True | Folge 1 | Zeile 391 | Das Denken soll man den Pferden überlassen, die haben den größeren Kopf.
True | Folge 2 | Zeile 467 | Wenn du irgendetwas verloren hast, nimm an, du hättest es einem Armen gegeben.
True | Folge 3 | Zeile 47 | Der Ellenbogen ist dem Munde nahe, dennoch kann man nicht selbst hineinbeißen.
True | Folge 4 | Zeile 347 | Wovon das Herz voll ist, davon geht der Mund über.
True | Folge 5 | Zeile 67 | Je verdorbener der Staat, desto mehr Gesetze hat er.
True | Folge 6 | Zeile 580 | Erinnere dich, Mensch, dass du aus Staub bist und zu Staub zurückkehren wirst.
True | Folge 7 | Zeile 768 | Wer den Aal hält bei dem Schwanz, dem bleibt er weder halb noch ganz.
True | Folge 8 | Zeile 220 | Schlechte Handwerker klagen über ihr Werkzeug.
True | Folge 9 | Zeile 81 | Wenn man durch Zweifel läuft, ist der Weg zum Himmel lang.
True | Folge A | Zeile 159 | Jede Dummheit leidet am Ekel vor sich selbst.
True | Folge B | Zeil

Alle zwölf Sätze sind bekannte Sprichwörter und Redensarten, und jeder Klartext passt Buchstabe für Buchstabe auf die Zahlenfolge und die Zeile. Das ist eine viel stärkere Probe als der Wörterbuchvergleich.

## Schritt 8: Wie sicher ist der Sieger?

Warum können wir dem Wörterbuchvergleich trauen? Wir sehen uns für `zahlenfolge5` die Punktzahlen **aller** Kandidaten an. Der Sieger hat 127 Punkte. Bei einem Buchstabensalat entstehen nur selten Wörter ab 3 Buchstaben, und fast nie lange.

In [ ]:
folge5 = lies_folge('beispieldaten/zahlenfolge5.txt')
alle = []
for nr in range(len(zeilen)):
    k = kandidat(folge5, anfaenge[nr])
    if k is not None:
        alle.append((bewerte(k), k))
alle.sort(reverse=True)

print('Kandidaten:', len(alle))
print('davon unter 50 Punkten:', sum(1 for p, k in alle if p < 50))
print('davon mit mindestens 100 Punkten:', sum(1 for p, k in alle if p >= 100))
for p, k in alle[:3]:
    print(p, k)

Kandidaten: 9848
davon unter 50 Punkten: 9333
davon mit mindestens 100 Punkten: 1
127 jeverdorbenerderstaatdestomehrgesetzehater
93 bmeerrnuneeucmodemaeennkindiuhidigeoleidet
93 abwdeooiiauvorendenwrnnenneseelhisfreielea


Fast alle Kandidaten bekommen weniger als 50 Punkte. Nur der Sieger erreicht 100 Punkte, der nächste Kandidat kommt auf 93. Dessen Buchstabensalat enthält nur ein paar zufällig passende Wörter. Je länger die Zahlenfolge ist, desto unwahrscheinlicher ist ein Zufall, weil ja *jeder* Buchstabe passen müsste. Kurze Folgen sind deshalb die kritischen. Im Zweifel entscheidet man, wie bei der Einsendung auch, mit dem Auge unter den ersten Kandidaten der Rangliste.

## Schritt 9: Gegenprobe – selbst verschlüsseln

Zum Schluss bauen wir Zaras Verfahren selbst nach. Zu einem Satz und einer Startzeile suchen wir mit `text.find(buchstabe, ab)` jeweils das **nächste** Vorkommen des Buchstabens und notieren die Abstände. Danach entschlüsseln wir mit `kandidat` und sehen, ob der Satz zurückkommt. Zaras Beispielfolge sieht anders aus, weil sie nicht immer den nächsten Buchstaben nimmt (Abstände wie 93 oder 58). Beides ist erlaubt: Die Folge muss nur zum Text passen.

In [ ]:
def verschluessele(satz, nr):
    # Zahlenfolge für den Satz, wenn Zeile nr (Index in zeilen) der Startpunkt ist; immer der nächste passende Buchstabe
    start = anfaenge[nr]
    folge = []
    pos = start - 1                           # Position des zuletzt gefundenen Buchstabens
    for b in saeubere(satz):
        neu = text.find(b, pos + 1)
        if neu == -1:
            return None                       # Buchstabe kommt nicht mehr vor
        folge.append(neu - pos)
        pos = neu
    return folge


meine_folge = verschluessele('Ohne Liebe keine Wahrheit', 5)
print('Meine Folge (Start in Zeile', dateizeilen[5], '):', meine_folge)
print('Entschlüsselt:', kandidat(meine_folge, anfaenge[5]))

for satz in ['Wer rastet, der rostet', 'Morgenstund hat Gold im Mund', 'Aller guten Dinge sind drei']:
    folge = verschluessele(satz, 4000)
    punkte_e, zeile_e, k_e, zweit_e = entschluessele(folge)
    print(satz, '(%d Buchstaben)' % len(folge))
    print('   Gefunden:', zerlege(k_e), '| Zeile', zeile_e, '(richtig: %d)' % dateizeilen[4000], '| Punkte', punkte_e, 'gegen', zweit_e)

Meine Folge (Start in Zeile 9 ): [20, 3, 8, 4, 26, 15, 8, 43, 1, 7, 1, 3, 41, 3, 147, 14, 19, 7, 4, 4, 17, 29]
Entschlüsselt: ohneliebekeinewahrheit
Wer rastet, der rostet (18 Buchstaben)
   Gefunden: m a s reden her e hinein | Zeile 125 (richtig: 4737) | Punkte 70 gegen 68
Morgenstund hat Gold im Mund (24 Buchstaben)
   Gefunden: morgen stund hat g old i m mund | Zeile 4737 (richtig: 4737) | Punkte 95 gegen 70
Aller guten Dinge sind drei (23 Buchstaben)
   Gefunden: aller guten dinge sind drei | Zeile 4737 (richtig: 4737) | Punkte 107 gegen 61


Der Beispielsatz wird bei der Probe wieder entschlüsselt. Bei den drei eigenen Sätzen, die wir ab Zeile 4737 der Datei verschlüsselt haben, zeigt sich die Grenze des Verfahrens: Die zwei längeren Sätze (24 und 23 Buchstaben) finden wir mit deutlichem Vorsprung. Beim kurzen Satz „Wer rastet, der rostet“ (18 Buchstaben) gewinnt dagegen ein Zufallstreffer (70 gegen 68 Punkte) und die richtige Zeile bleibt unentdeckt. Das passt zu dem, was wir bei `zahlenfolge5` gesehen haben: Bei kurzen Folgen reicht ein Zufall, um den Wörterbuchvergleich zu täuschen. Dann hilft nur, die ersten Plätze der Rangliste anzusehen. Zara hat bei den Aufgabenfolgen 22 bis 65 Buchstaben gewählt, das reicht für ein eindeutiges Ergebnis. Auch bei Zaras Folgen kommen große Abstände vor (wie 93 oder 295), weil sie im Buch weiter hinten abgezählt hat. Für unser Programm macht das keinen Unterschied.

## Was haben wir gelernt?

- Einen Suchraum, der nur ein paar tausend Möglichkeiten hat (die Zeilen des Buches), probiert man einfach **vollständig** durch.
- Eine **Bewertungsfunktion** macht aus „finde den richtigen“ ein „nimm den besten“. Wichtig ist, dass sie die richtige Antwort klar vom Zufall trennt.
- Die Zerlegung in Wörter ist ein Fall für **dynamische Programmierung**: kleinere Reste einmal berechnen und wiederverwenden. Zusammen mit einem **Wörterbuch als Menge** ist das schnell genug.

## Zum Schluss: Warum kein MiniZinc?

MiniZinc ist stark, wenn man Bedingungen angibt und der Solver eine passende Belegung sucht. Hier wäre das Modell klein: eine Variable `s` für die Startstelle und für jeden Buchstaben des Satzes die Bedingung `text[s + summe[i] - 1] = satz[i]`. Wir haben dieses Modell mit Gecode bei allen 12 Folgen ausprobiert. Es findet jeweils die Startstelle, die wir mit Python gefunden haben, und braucht pro Folge ungefähr 8 bis 12 Sekunden. Das ist aber **keine Lösung der Aufgabe**: Das Modell muss den Satz bereits kennen. Die eigentliche Schwierigkeit (welcher Kandidat ist ein deutscher Satz?) ist eine Wörterbuchfrage und keine Bedingung, die man sinnvoll in MiniZinc formuliert. Dazu müsste man das ganze Wörterbuch hineinschreiben und jeden Kandidaten in Wörter zerlegen. Außerdem ist der Text mit 484 440 Buchstaben für eine Datendatei sehr groß, fast die ganze Zeit geht für das Einlesen drauf. Deshalb haben wir hier kein MiniZinc-Modell aufgenommen: Die direkte Python-Lösung ist einfacher und viel schneller.